## 1. Brevitas Import, FINN Export 
En primer lugar importamos el modelo generado con Brevitas y lo exportamos a un modelo de onnx finn.

In [1]:
import json
import numpy as np
import os
import shutil
import warnings
from copy import deepcopy
from distutils.dir_util import copy_tree
from finn.util.basic import make_build_dir
from finn.util.visualization import showInNetron
import finn.builder.build_dataflow as build
import finn.builder.build_dataflow_config as build_cfg
import onnx
from finn.util.test import get_test_model_trained
from qonnx.core.modelwrapper import ModelWrapper
from qonnx.transformation.infer_shapes import InferShapes
from qonnx.transformation.fold_constants import FoldConstants
from qonnx.transformation.infer_datatypes import InferDataTypes
from qonnx.transformation.general import (
    ApplyConfig,
    GiveReadableTensorNames,
    GiveUniqueNodeNames,
    RemoveStaticGraphInputs,
    RemoveUnusedTensors,
)
import brevitas.onnx as bo
import brevitas.nn as qnn
from brevitas.quant import Int8Bias
from brevitas.core.restrict_val import RestrictValueType
import torch 
import torch.nn as nn
import torch.nn.functional as F
from qonnx.util.cleanup import cleanup_model
from qonnx.util.config import extract_model_config_to_json
from shutil import copy

import finn.transformation.fpgadataflow.convert_to_hls_layers as to_hls
import finn.transformation.streamline.absorb as absorb
from finn.analysis.fpgadataflow.dataflow_performance import dataflow_performance
from finn.analysis.fpgadataflow.exp_cycles_per_layer import exp_cycles_per_layer
from finn.analysis.fpgadataflow.hls_synth_res_estimation import hls_synth_res_estimation
from finn.analysis.fpgadataflow.op_and_param_counts import (
    aggregate_dict_keys,
    op_and_param_counts,
)
from finn.analysis.fpgadataflow.res_estimation import (
    res_estimation,
    res_estimation_complete,
)
from finn.builder.build_dataflow_config import (
    DataflowBuildConfig,
    DataflowOutputType,
    ShellFlowType,
    VerificationStepType,
)
from finn.core.onnx_exec import execute_onnx
from finn.core.rtlsim_exec import rtlsim_exec
from finn.core.throughput_test import throughput_test_rtlsim
from finn.transformation.fpgadataflow.annotate_cycles import AnnotateCycles
from finn.transformation.fpgadataflow.compile_cppsim import CompileCppSim
from finn.transformation.fpgadataflow.create_dataflow_partition import (
    CreateDataflowPartition,
)
from finn.transformation.fpgadataflow.create_stitched_ip import CreateStitchedIP
from finn.transformation.fpgadataflow.derive_characteristic import (
    DeriveCharacteristic,
    DeriveFIFOSizes,
)
from finn.transformation.fpgadataflow.insert_tlastmarker import InsertTLastMarker
from finn.transformation.fpgadataflow.hlssynth_ip import HLSSynthIP
from finn.transformation.fpgadataflow.insert_dwc import InsertDWC
from finn.transformation.fpgadataflow.insert_fifo import InsertFIFO
from finn.transformation.fpgadataflow.make_pynq_driver import MakePYNQDriver
from finn.transformation.fpgadataflow.make_zynq_proj import ZynqBuild
from finn.transformation.fpgadataflow.prepare_cppsim import PrepareCppSim
from finn.transformation.fpgadataflow.prepare_ip import PrepareIP
from finn.transformation.fpgadataflow.prepare_rtlsim import PrepareRTLSim
from finn.transformation.fpgadataflow.replace_verilog_relpaths import (
    ReplaceVerilogRelPaths,
)
from finn.transformation.fpgadataflow.set_exec_mode import SetExecMode
from finn.transformation.fpgadataflow.set_fifo_depths import (
    InsertAndSetFIFODepths,
    RemoveShallowFIFOs,
    SplitLargeFIFOs,
)
from finn.transformation.fpgadataflow.set_folding import SetFolding
from finn.transformation.fpgadataflow.synth_ooc import SynthOutOfContext
from finn.transformation.fpgadataflow.vitis_build import VitisBuild
from finn.transformation.move_reshape import RemoveCNVtoFCFlatten
from finn.transformation.qonnx.convert_qonnx_to_finn import ConvertQONNXtoFINN
from finn.transformation.qonnx.quant_act_to_multithreshold import (
    default_filter_function_generator,
)
from finn.transformation.streamline import Streamline
from finn.transformation.streamline.reorder import MakeMaxPoolNHWC
from finn.util.basic import (
    get_rtlsim_trace_depth,
    pyverilate_get_liveness_threshold_cycles,
)
from finn.util.pyverilator import verilator_fifosim
from finn.util.test import execute_parent

#AÑADIDO
from brevitas.inject.enum import ScalingImplType

In [3]:
import json
import numpy as np
import os
import shutil
import warnings
import torch 
import torch.nn as nn
import brevitas.nn as qnn
import brevitas.onnx as bo
from qonnx.core.datatype import DataType
from qonnx.core.modelwrapper import ModelWrapper
import finn.builder.build_dataflow as build
import finn.builder.build_dataflow_config as build_cfg

# 1. LIMPIEZA TOTAL
final_output_dir = "output_final_ok"
if os.path.exists(final_output_dir):
    shutil.rmtree(final_output_dir)
    print("¡Carpeta antigua fulminada! Empezamos limpios.")

# 2. CONFIGURACIÓN ZCU104
cfg = build.DataflowBuildConfig(
    output_dir          = final_output_dir,
    folding_config_file = "folding_config_3cnv.json", 
    auto_fifo_depths    = False,
    mvau_wwidth_max     = 36,
    stitched_ip_gen_dcp = True,
    split_large_fifos   = True,
    synth_clk_period_ns = 10.0,
    board               = "Pynq-Z2",  
    default_mem_mode    = "decoupled", # Modo super optimizado
    shell_flow_type     = build_cfg.ShellFlowType.VIVADO_ZYNQ,
    generate_outputs=[
        build_cfg.DataflowOutputType.ESTIMATE_REPORTS,
        build_cfg.DataflowOutputType.STITCHED_IP,
        build_cfg.DataflowOutputType.RTLSIM_PERFORMANCE,
        build_cfg.DataflowOutputType.OOC_SYNTH,
    ]
)
# 3. RED Y EXPORTACIÓN
class QMLPModel(nn.Module):
    def __init__(self, latent_features, dropout_rate=0.1, num_classes=2, input_w=8, output_w=8, weigth_w=8):
        super(QMLPModel, self).__init__()
        self.quant_input = qnn.QuantIdentity(bit_width=input_w, return_quant_tensor=True, min_val=0, max_val=1)
        
        # CAPA 1
        self.fc1 = qnn.QuantLinear(3 * 16 * 12, 256, bias=True, weight_bit_width=weigth_w, return_quant_tensor=True)
        self.relu1 = qnn.QuantReLU(bit_width=weigth_w, return_quant_tensor=True)
        self.dropout1 = qnn.QuantDropout(dropout_rate)
        
        # CAPA 2
        self.fc2 = qnn.QuantLinear(256, 128, bias=True, weight_bit_width=weigth_w, return_quant_tensor=True)
        self.relu2 = qnn.QuantReLU(bit_width=weigth_w, return_quant_tensor=True)
        self.dropout2 = qnn.QuantDropout(dropout_rate)

        # CAPA 3
        self.fx = qnn.QuantLinear(128, latent_features, bias=True, weight_bit_width=weigth_w, return_quant_tensor=True)
        self.relu3 = qnn.QuantReLU(bit_width=weigth_w, return_quant_tensor=True)

        # CAPA 4 (Salida)
        self.predictor = qnn.QuantLinear(latent_features, num_classes, weight_bit_width=weigth_w, bias=True, return_quant_tensor=False)
        # self.quant_output = qnn.QuantIdentity(bit_width=output_w, return_quant_tensor=True, min_val=-1.0, max_val=1.0)
        self.quant_output = qnn.QuantHardTanh(bit_width=output_w, min_val=-1.0, max_val=1.0, return_quant_tensor=True)
        
    def forward(self, x):
        x = self.quant_input(x)
        x = self.fc1(x)
        x = self.relu1(x)
        x = self.dropout1(x)
        x = self.fc2(x)
        x = self.relu2(x)
        x = self.dropout2(x)
        x = self.fx(x)
        x = self.relu3(x)
        x = self.predictor(x)
        x = self.quant_output(x)
        return x

model = QMLPModel(latent_features=16, dropout_rate=0.05, num_classes=2, input_w=8, output_w=8, weigth_w=8)

# Como usamos el script de extracción limpio, cargamos el diccionario directamente
estado_original = torch.load("state_dict_quantized_last.pth", map_location=torch.device('cpu'))
model.load_state_dict(estado_original)
model.eval()

model_file_name = "neuralpilot-mlp-ready.onnx"
bo.export_finn_onnx(model, (1, 3 * 16 * 12), model_file_name)

model_wrapper = ModelWrapper(model_file_name)
model_wrapper.set_tensor_datatype(model_wrapper.graph.input[0].name, DataType["UINT8"]) 
model_wrapper.set_tensor_datatype(model_wrapper.graph.output[0].name, DataType["INT8"]) 
model_wrapper.save(model_file_name)
print("¡Fase 1 completada: ONNX exportado y limpio!")

¡Fase 1 completada: ONNX exportado y limpio!


/home/ruben/finn/deps/qonnx/src/qonnx/core/modelwrapper.py:93: UserWarning: Some old-style domain attributes were automatically converted to new-style,
                i.e. domain=finn to domain=qonnx.custom_op.<general|fpgadataflow|...>
  warnings.warn(


## 2. Tidy-Up: Limpieza del Grafo

In [4]:
from finn.transformation.streamline import Streamline
import finn.transformation.streamline.absorb as absorb
from finn.transformation.streamline.reorder import MoveScalarLinearPastInvariants
from qonnx.transformation.infer_data_layouts import InferDataLayouts
from qonnx.transformation.general import RemoveUnusedTensors
import finn.transformation.fpgadataflow.convert_to_hls_layers as to_hls
from finn.transformation.fpgadataflow.create_dataflow_partition import CreateDataflowPartition
from qonnx.custom_op.registry import getCustomOp

model_wrapper = ModelWrapper("neuralpilot-mlp-ready.onnx")

# Streamline
model_wrapper = model_wrapper.transform(MoveScalarLinearPastInvariants())
model_wrapper = model_wrapper.transform(absorb.AbsorbSignBiasIntoMultiThreshold())
model_wrapper = model_wrapper.transform(Streamline())
model_wrapper = model_wrapper.transform(InferDataLayouts())
model_wrapper = model_wrapper.transform(RemoveUnusedTensors())

# Conversión HLS
model_wrapper = model_wrapper.transform(to_hls.InferQuantizedMatrixVectorActivation("decoupled"))
model_wrapper = model_wrapper.transform(to_hls.InferThresholdingLayer())
model_wrapper = model_wrapper.transform(InferDataLayouts())

# Aislamiento
parent_model = model_wrapper.transform(CreateDataflowPartition(partition_model_dir=cfg.output_dir + "/intermediate_models/supported_op_partitions"))
sdp_node = getCustomOp(parent_model.get_nodes_by_op_type("StreamingDataflowPartition")[0])
model_wrapper = ModelWrapper(sdp_node.get_nodeattr("model"))
model_wrapper.save("neuralpilot-mlp-ready.onnx")

print("¡Fase 2 completada: Modelo convertido a HLS!")

/home/ruben/finn/deps/qonnx/src/qonnx/transformation/infer_data_layouts.py:124: UserWarning: Assuming 2D input is NC
  warnings.warn("Assuming 2D input is NC")


¡Fase 2 completada: Modelo convertido a HLS!


## 3. Streamlining: Matando a los "FLOATS"

In [5]:
from qonnx.transformation.general import GiveUniqueNodeNames, ApplyConfig, GiveReadableTensorNames
from finn.transformation.fpgadataflow.insert_dwc import InsertDWC
from finn.transformation.fpgadataflow.insert_fifo import InsertFIFO
from finn.transformation.fpgadataflow.set_fifo_depths import RemoveShallowFIFOs
from finn.transformation.fpgadataflow.insert_tlastmarker import InsertTLastMarker
import json
from qonnx.core.modelwrapper import ModelWrapper

model_wrapper = ModelWrapper("neuralpilot-mlp-ready.onnx")
model_wrapper = model_wrapper.transform(GiveUniqueNodeNames())

fc_layers = model_wrapper.get_nodes_by_op_type("MatrixVectorActivation")
thresh_layers = model_wrapper.get_nodes_by_op_type("Thresholding_Batch")

folding_config = {"Defaults": {"ram_style": "distributed", "resType": "dsp"}}

# EL FOLDING CORRECTO (Ahora con 4 capas FC)
# EL FOLDING CORRECTO (Usando BRAM para la capa inicial gigante)
# "auto" o "block" forzará el uso de BRAMs en lugar de devorar las LUTs
if len(fc_layers) > 0: folding_config[fc_layers[0].name] = {"PE": 1, "SIMD": 1, "ram_style": "auto", "resType": "dsp"}
if len(fc_layers) > 1: folding_config[fc_layers[1].name] = {"PE": 1, "SIMD": 1, "ram_style": "auto", "resType": "dsp"}
if len(fc_layers) > 2: folding_config[fc_layers[2].name] = {"PE": 1, "SIMD": 1, "ram_style": "distributed", "resType": "dsp"}
if len(fc_layers) > 3: folding_config[fc_layers[3].name] = {"PE": 1, "SIMD": 1, "ram_style": "distributed", "resType": "dsp"} # <-- Nueva capa

if len(thresh_layers) > 0: folding_config[thresh_layers[0].name] = {"PE": 1, "ram_style": "distributed"}
if len(thresh_layers) > 1: folding_config[thresh_layers[1].name] = {"PE": 1, "ram_style": "distributed"}
if len(thresh_layers) > 2: folding_config[thresh_layers[2].name] = {"PE": 1, "ram_style": "distributed"}
if len(thresh_layers) > 3: folding_config[thresh_layers[3].name] = {"PE": 1, "ram_style": "distributed"} # <-- Por si acaso hay una 4ª
with open("folding_config_3cnv.json", "w") as f:
    json.dump(folding_config, f, indent=4)

model_wrapper = model_wrapper.transform(ApplyConfig("folding_config_3cnv.json"))

# Insertamos FIFOS, conversores y TLast
model_wrapper = model_wrapper.transform(InsertDWC())
model_wrapper = model_wrapper.transform(InsertFIFO(create_shallow_fifos=True))
model_wrapper = model_wrapper.transform(InsertTLastMarker(dynamic=False))
model_wrapper = model_wrapper.transform(GiveUniqueNodeNames())
model_wrapper = model_wrapper.transform(GiveReadableTensorNames())

# Parcheamos el JSON SOLO para los FIFOs (que sí aceptan estos atributos)
fifos = model_wrapper.get_nodes_by_op_type("StreamingFIFO")
for f_node in fifos: 
    folding_config[f_node.name] = {"ram_style": "auto", "depth": 2}

with open("folding_config_3cnv.json", "w") as f: 
    json.dump(folding_config, f, indent=4)
    
model_wrapper = model_wrapper.transform(ApplyConfig("folding_config_3cnv.json"))
model_wrapper = model_wrapper.transform(RemoveShallowFIFOs())

model_wrapper.save("neuralpilot-mlp-ready.onnx")
print("¡Fase 3 completada: Folding extremo de ZCU104 aplicado!")

¡Fase 3 completada: Folding extremo de ZCU104 aplicado!


/home/ruben/finn/deps/qonnx/src/qonnx/transformation/general.py:322: UserWarning: 
No HW configuration for nodes: TLastMarker_0
  warnings.warn("\nNo HW configuration for nodes: " + ", ".join(missing_configurations))


## 4: Aplicar la Configuración de Folding

In [6]:
from finn.transformation.fpgadataflow.prepare_ip import PrepareIP
from finn.transformation.fpgadataflow.hlssynth_ip import HLSSynthIP
from finn.transformation.fpgadataflow.replace_verilog_relpaths import ReplaceVerilogRelPaths
from finn.transformation.fpgadataflow.create_stitched_ip import CreateStitchedIP
from finn.transformation.fpgadataflow.synth_ooc import SynthOutOfContext
from finn.transformation.fpgadataflow.annotate_cycles import AnnotateCycles
from finn.analysis.fpgadataflow.dataflow_performance import dataflow_performance
from distutils.dir_util import copy_tree

model_wrapper = ModelWrapper("neuralpilot-mlp-ready.onnx")

print("1/3 Preparando IPs y sintetizando C++ (Esto tarda un poco)...")
model_wrapper = model_wrapper.transform(PrepareIP(cfg._resolve_fpga_part(), cfg._resolve_hls_clk_period()))
model_wrapper = model_wrapper.transform(HLSSynthIP())
model_wrapper = model_wrapper.transform(ReplaceVerilogRelPaths())

print("2/3 Cosido del IP (Stitching)...")
model_wrapper = model_wrapper.transform(CreateStitchedIP(cfg._resolve_fpga_part(), cfg.synth_clk_period_ns, vitis=cfg.stitched_ip_gen_dcp))
copy_tree(model_wrapper.get_metadata_prop("vivado_stitch_proj"), cfg.output_dir + "/stitched_ip")

print("3/3 Síntesis de Vivado y calculo de rendimiento...")
model_wrapper = model_wrapper.transform(SynthOutOfContext(part=cfg._resolve_fpga_part(), clk_period_ns=cfg.synth_clk_period_ns))
model_wrapper = model_wrapper.transform(AnnotateCycles())
model_wrapper.save("neuralpilot-mlp-ready.onnx")

# --- REPORTE ---
report_dir = cfg.output_dir + "/report"
os.makedirs(report_dir, exist_ok=True)
ooc_res_dict = eval(model_wrapper.get_metadata_prop("res_total_ooc_synth"))
estimate_network_performance = model_wrapper.analysis(dataflow_performance)
est_fps = (float(ooc_res_dict["fmax_mhz"]) * (10**6)) / estimate_network_performance["max_cycles"]
ooc_res_dict["estimated_throughput_fps"] = est_fps

with open(report_dir + "/ooc_synth_and_timing.json", "w") as f:
    json.dump(ooc_res_dict, f, indent=2)

print("\n" + "="*40 + "\n REPORTE FINAL OOC \n" + "="*40)
res = ooc_res_dict.get("total", ooc_res_dict)
print(f" LUTs:  {res.get('LUT', 0)}")
print(f" FFs:   {res.get('FF', 0)}")
print(f" DSPs:  {res.get('DSP', 0)}")
print(f" BRAMs: {res.get('BRAM_18K', 0)} (18K)")
print(f" URAMs: {res.get('URAM', 0)}")
print(f"\n Frecuencia: {res.get('fmax_mhz'):.2f} MHz")
print(f" FPS:        {est_fps:.2f} frames/sec")

1/3 Preparando IPs y sintetizando C++ (Esto tarda un poco)...
2/3 Cosido del IP (Stitching)...
3/3 Síntesis de Vivado y calculo de rendimiento...
ap_clk
xc7z020clg400-1
10.000000
0

****** Vivado v2022.2 (64-bit)
  **** SW Build 3671981 on Fri Oct 14 04:59:54 MDT 2022
  **** IP Build 3669848 on Fri Oct 14 08:30:02 MDT 2022
    ** Copyright 1986-2022 Xilinx, Inc. All Rights Reserved.

source finn_design_wrapper.tcl
# set fpga_part "xc7z020clg400-1"
# set origin_dir "."
# if { [info exists ::origin_dir_loc] } {
#   set origin_dir $::origin_dir_loc
# }
# variable script_file
# set script_file "vivadocompile.tcl"
# proc help {} {
#   variable script_file
#   puts "\nDescription:"
#   puts "Recreate a Vivado project from this script. The created project will be"
#   puts "functionally equivalent to the original project for which this script was"
#   puts "generated. The script contains commands for creating a project, filesets,"
#   puts "runs, adding/importing sources and setting propertie

/home/ruben/finn/deps/oh-my-xilinx/vivadoprojgen.sh:30: no matches found: ../*.vhd
/home/ruben/finn/deps/oh-my-xilinx/vivadoprojgen.sh:32: no matches found: ../*.h
/home/ruben/finn/deps/oh-my-xilinx/vivadoprojgen.sh:33: no matches found: ../*.xdc
/home/ruben/finn/deps/oh-my-xilinx/vivadoprojgen.sh:35: no matches found: ../*.sv
cat: finn_design_wrapper.xdc: input file is output file
